<a href="https://colab.research.google.com/github/deeppatel710/MiniWarehouseWorkshop/blob/main/MiniWarehouseWorkshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/YOUR_USERNAME/pyspark-warehouse-workshop.git

In [ ]:
!cd MiniWarehouseWorkshop

In [ ]:
!ls

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MiniWarehouseWorkshop") \
    .getOrCreate()

In [42]:
users = spark.read.csv("/content/MiniWarehouseWorkshop/data/users.csv", header=True, inferSchema=True)
user_fields = spark.read.csv("/content/MiniWarehouseWorkshop/data/user_fields.csv", header=True, inferSchema=True)
products = spark.read.csv("/content/MiniWarehouseWorkshop/data/products.csv", header=True, inferSchema=True)
product_categories = spark.read.csv("/content/MiniWarehouseWorkshop/data/product_categories.csv", header=True, inferSchema=True)
orders = spark.read.csv("/content/MiniWarehouseWorkshop/data/orders.csv", header=True, inferSchema=True)
order_items = spark.read.csv("/content/MiniWarehouseWorkshop/data/order_items.csv", header=True, inferSchema=True)

In [43]:
users.printSchema()
users.show(10)

root
 |-- user_id: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- field_id: string (nullable = true)
 |-- created_at: timestamp (nullable = true)

+-------+--------------------+----------+-------+--------+-------------------+
|user_id|           user_name|      city|country|field_id|         created_at|
+-------+--------------------+----------+-------+--------+-------------------+
|   U001| Neil deGrasse Tyson|  New York|    USA|     F01|2023-03-14 09:22:00|
|   U001| Neil deGrasse Tyson|  New York|    USA|     F01|2023-03-14 09:22:00|
|   U002|          Kip Thorne|  Pasadena|    USA|     F01|2022-07-08 14:31:00|
|   U003|           Brian Cox|Manchester|     UK|     F02|2023-11-19 08:45:00|
|   U004|         Sara Seager| Cambridge|    USA|     F03|2022-01-25 16:10:00|
|   U005|Subrahmanyan Chan...|   Chicago|    USA|     F01|2024-02-11 11:00:00|
|   U006|          Vera Rubin|Washington|  

In [ ]:
products.printSchema()
products.show(10)

In [ ]:

product_categories.printSchema()
product_categories.show(10)

In [29]:
orders.printSchema()
orders.show(10)

root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- order_date: date (nullable = true)

+--------+-------+----------+
|order_id|user_id|order_date|
+--------+-------+----------+
|    O001|   U001|2024-01-05|
|    O002|   U002|2024-01-08|
|    O003|   U001|2024-01-12|
|    O004|   U003|2024-01-20|
|    O005|   U001|2024-02-02|
|    O006|   U004|2024-02-10|
|    O007|   U002|2024-02-15|
|    O008|   U005|2024-02-20|
|    O009|   U003|2024-03-01|
|    O010|   U006|2024-03-05|
+--------+-------+----------+



In [28]:
order_items.printSchema()
order_items.show(10)

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)

+--------+----------+--------+
|order_id|product_id|quantity|
+--------+----------+--------+
|    O001|      P001|       1|
|    O002|      P010|       1|
|    O003|      P005|       2|
|    O004|      P003|       3|
|    O005|      P002|       1|
|    O006|      P006|       1|
|    O007|      P004|       2|
|    O008|      P001|       1|
|    O009|      P008|       1|
|    O010|      P006|       2|
+--------+----------+--------+



### GOAL: Calculate Revenue per Field per Month

OLTP way of doing this:

In [47]:
from pyspark.sql.functions import col, month, year, sum

oltp_query = (
    order_items
    .join(products, "product_id")
    .join(orders, "order_id")
    .join(users, "user_id")
    .join(user_fields, "field_id")
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
    .withColumn("revenue", col("quantity") * col("unit_price"))
    .groupBy("field_name", "year", "month")
    .agg(sum("revenue").alias("total_revenue"))
    .orderBy("field_name", "year", "month")
)

oltp_query.show()


+-----------------+----+-----+-------------+
|       field_name|year|month|total_revenue|
+-----------------+----+-----+-------------+
|     Astrophysics|2024|    1|      3050000|
|     Astrophysics|2024|    2|      2090000|
|     Astrophysics|2024|    3|      1800000|
| Particle Physics|2024|    1|       135000|
| Particle Physics|2024|    3|      1250000|
|Planetary Science|2024|    2|       300000|
+-----------------+----+-----+-------------+



# OLAP WAY : THE ONLY RIGHT WAY

# Cleaning first

In [52]:
users = users.dropna()

users = users.withColumn(
    "created_at",
    col("created_at").cast("date")
)

users_clean_df = users.dropDuplicates(["user_id"])

In [53]:
users_clean_df.show(10)

+-------+--------------------+-----------+-------+--------+----------+
|user_id|           user_name|       city|country|field_id|created_at|
+-------+--------------------+-----------+-------+--------+----------+
|   U001| Neil deGrasse Tyson|   New York|    USA|     F01|2023-03-14|
|   U002|          Kip Thorne|   Pasadena|    USA|     F01|2022-07-08|
|   U003|           Brian Cox| Manchester|     UK|     F02|2023-11-19|
|   U004|         Sara Seager|  Cambridge|    USA|     F03|2022-01-25|
|   U005|Subrahmanyan Chan...|    Chicago|    USA|     F01|2024-02-11|
|   U006|          Vera Rubin| Washington|    USA|     F01|2023-06-30|
|   U007|     Stephen Hawking|  Cambridge|     UK|     F01|2022-09-03|
|   U008|          Carl Sagan|     Ithaca|    USA|     F03|2023-12-20|
|   U009|Jocelyn Bell Burnell|     Oxford|     UK|     F01|2024-04-05|
|   U010|         Andrea Ghez|Los Angeles|    USA|     F01|2022-11-12|
+-------+--------------------+-----------+-------+--------+----------+



In [49]:
from pyspark.sql.functions import col

orders = orders.dropna()

orders = orders.withColumn(
    "order_date",
    col("order_date").cast("date")
)

orders_df = orders.dropDuplicates(["order_id"])

In [54]:
orders_df.show(10)

+--------+-------+----------+
|order_id|user_id|order_date|
+--------+-------+----------+
|    O001|   U001|2024-01-05|
|    O002|   U002|2024-01-08|
|    O003|   U001|2024-01-12|
|    O004|   U003|2024-01-20|
|    O005|   U001|2024-02-02|
|    O006|   U004|2024-02-10|
|    O007|   U002|2024-02-15|
|    O008|   U005|2024-02-20|
|    O009|   U003|2024-03-01|
|    O010|   U006|2024-03-05|
+--------+-------+----------+



# We will create:

# ⭐ dim_user
# ⭐ dim_product
# ⭐ dim_date
# ⭐ fact_sales

In [62]:
from pyspark.sql.functions import concat_ws, monotonically_increasing_id, split

dim_user = (
    users
    .join(user_fields, "field_id")
    .select("user_id", "user_name", "field_name", "country")
    .withColumn("user_sk", monotonically_increasing_id())
)
dim_user.show(10)

+-------+--------------------+-----------------+-------+-------+
|user_id|           user_name|       field_name|country|user_sk|
+-------+--------------------+-----------------+-------+-------+
|   U001| Neil deGrasse Tyson|     Astrophysics|    USA|      0|
|   U001| Neil deGrasse Tyson|     Astrophysics|    USA|      1|
|   U002|          Kip Thorne|     Astrophysics|    USA|      2|
|   U003|           Brian Cox| Particle Physics|     UK|      3|
|   U004|         Sara Seager|Planetary Science|    USA|      4|
|   U005|Subrahmanyan Chan...|     Astrophysics|    USA|      5|
|   U006|          Vera Rubin|     Astrophysics|    USA|      6|
|   U006|          Vera Rubin|     Astrophysics|    USA|      7|
|   U006|          Vera Rubin|     Astrophysics|    USA|      8|
|   U007|     Stephen Hawking|     Astrophysics|     UK|      9|
+-------+--------------------+-----------------+-------+-------+
only showing top 10 rows


In [63]:
dim_product = (
    products
    .join(product_categories, "category_id")
    .select("product_id", "product_name", "category_name", "unit_price")
    .withColumn("product_sk", monotonically_increasing_id())
)
dim_product.show(10)

+----------+--------------------+--------------------+----------+----------+
|product_id|        product_name|       category_name|unit_price|product_sk|
+----------+--------------------+--------------------+----------+----------+
|      P001|High-Resolution S...|     Instrumentation|    150000|         0|
|      P002|Radio Telescope R...|     Instrumentation|    850000|         1|
|      P003|  CCD Imaging Sensor|              Optics|     45000|         2|
|      P004|Cryogenic Cooling...|       Lab Equipment|    120000|         3|
|      P005|Supercomputer Com...|           Computing|    250000|         4|
|      P006|Adaptive Optics M...|              Optics|    300000|         5|
|      P008|Dark Matter Detec...|Experimental Physics|   1250000|         6|
|      P010|Gravitational Wav...|Experimental Physics|   1750000|         7|
+----------+--------------------+--------------------+----------+----------+



In [65]:
dim_date = (
    orders
    .select("order_date")
    .dropDuplicates()
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
    .withColumn("date_sk", monotonically_increasing_id())
)
dim_date.show(10)

+----------+----+-----+-------+
|order_date|year|month|date_sk|
+----------+----+-----+-------+
|2024-03-05|2024|    3|      0|
|2024-02-15|2024|    2|      1|
|2024-02-10|2024|    2|      2|
|2024-01-05|2024|    1|      3|
|2024-01-12|2024|    1|      4|
|2024-02-20|2024|    2|      5|
|2024-03-01|2024|    3|      6|
|2024-02-02|2024|    2|      7|
|2024-01-08|2024|    1|      8|
|2024-01-20|2024|    1|      9|
+----------+----+-----+-------+



In [67]:
fact_sales = (
    order_items
    .join(orders, "order_id")
    .join(dim_user, "user_id")
    .join(dim_product, "product_id")
    .join(dim_date, "order_date")
    .select(
        "order_id",
        "user_sk",
        "product_sk",
        "date_sk",
        "quantity",
        (col("quantity") * col("unit_price")).alias("revenue")
    )
)
fact_sales.show(10)

+--------+-------+----------+-------+--------+-------+
|order_id|user_sk|product_sk|date_sk|quantity|revenue|
+--------+-------+----------+-------+--------+-------+
|    O001|      1|         0|      3|       1| 150000|
|    O001|      0|         0|      3|       1| 150000|
|    O002|      2|         7|      8|       1|1750000|
|    O003|      1|         4|      4|       2| 500000|
|    O003|      0|         4|      4|       2| 500000|
|    O004|      3|         2|      9|       3| 135000|
|    O005|      1|         1|      7|       1| 850000|
|    O005|      0|         1|      7|       1| 850000|
|    O006|      4|         5|      2|       1| 300000|
|    O007|      2|         3|      1|       2| 240000|
+--------+-------+----------+-------+--------+-------+
only showing top 10 rows


# Analytics in Warehouse Layer

In [68]:
analytics_query = (
    fact_sales
    .join(dim_user, "user_sk")
    .join(dim_date, "date_sk")
    .groupBy("field_name", "year", "month")
    .sum("revenue")
)

analytics_query.show()


+-----------------+----+-----+------------+
|       field_name|year|month|sum(revenue)|
+-----------------+----+-----+------------+
|     Astrophysics|2024|    2|     2090000|
| Particle Physics|2024|    3|     1250000|
| Particle Physics|2024|    1|      135000|
|     Astrophysics|2024|    1|     3050000|
|Planetary Science|2024|    2|      300000|
|     Astrophysics|2024|    3|     1800000|
+-----------------+----+-----+------------+



# Denormalize Further (BI Layer)

In [69]:
analytics_flat = (
    fact_sales
    .join(dim_user, "user_sk")
    .join(dim_product, "product_sk")
    .join(dim_date, "date_sk")
)

analytics_flat.show()


+-------+----------+-------+--------+--------+-------+-------+--------------------+-----------------+-------+----------+--------------------+--------------------+----------+----------+----+-----+
|date_sk|product_sk|user_sk|order_id|quantity|revenue|user_id|           user_name|       field_name|country|product_id|        product_name|       category_name|unit_price|order_date|year|month|
+-------+----------+-------+--------+--------+-------+-------+--------------------+-----------------+-------+----------+--------------------+--------------------+----------+----------+----+-----+
|      3|         0|      1|    O001|       1| 150000|   U001| Neil deGrasse Tyson|     Astrophysics|    USA|      P001|High-Resolution S...|     Instrumentation|    150000|2024-01-05|2024|    1|
|      3|         0|      0|    O001|       1| 150000|   U001| Neil deGrasse Tyson|     Astrophysics|    USA|      P001|High-Resolution S...|     Instrumentation|    150000|2024-01-05|2024|    1|
|      8|         7|

# Now analytics becomes:

In [70]:
analytics_flat.groupBy("category_name", "year").sum("revenue").show()


+--------------------+----+------------+
|       category_name|year|sum(revenue)|
+--------------------+----+------------+
|       Lab Equipment|2024|      240000|
|Experimental Physics|2024|     3000000|
|              Optics|2024|     2235000|
|           Computing|2024|     1000000|
|     Instrumentation|2024|     2150000|
+--------------------+----+------------+



# by country:

In [72]:
analytics_flat.groupBy("country", "year").sum("revenue").show()


+-------+----+------------+
|country|year|sum(revenue)|
+-------+----+------------+
|    USA|2024|     7240000|
|     UK|2024|     1385000|
+-------+----+------------+



# DO IT YOURSELF:
Question 1: Which research field generated the most revenue overall?

Question 2: Who are the top 3 scientists by total revenue?